# Davinci — Multi-source Pretraining Corpus Merger

Merge separately audited pretraining sources into one deterministic, deduplicated corpus. Every source must have an immutable revision, an approved provenance/license manifest, and a verified artifact hash. This notebook never treats a missing license as permission and never approves a source automatically.

## 1. Install and import the canonical pipeline

In [ ]:
!pip -q install tqdm

In [ ]:
import hashlib, json, sqlite3, subprocess, sys
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive

drive.mount('/content/drive')
REPO_DIR=Path('/content/Davinci')
if not (REPO_DIR/'.git').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/keyhmoham/Davinci.git',str(REPO_DIR)],check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only'],check=True)
sys.path.insert(0,str(REPO_DIR))
from src.davinci.data_pipeline import iter_jsonl, load_dataset_manifest, validate_dataset_manifest, verify_dataset_artifact, sha256_file
from src.davinci.data_audit import normalize_persian_text, content_sha256
from src.davinci.near_dedup import find_near_duplicate_pairs
from itertools import islice

## 2. Select source artifacts

Run the single-source dataset pipeline once for each candidate and review each resulting manifest. Do not include an already merged output as an input to itself. Add only sources intended for pretraining.

In [ ]:
ROOT=Path('/content/drive/MyDrive/Davinci/data_pipeline/processed')
SOURCES=[
    {'key':'naab','data':ROOT/'naab.jsonl','manifest':ROOT/'naab_manifest.json'},
    {'key':'persian_blog','data':ROOT/'persian_blog.jsonl','manifest':ROOT/'persian_blog_manifest.json'},
    # Add additional reviewed source artifacts here.
]
OUTPUT=ROOT/'persian_corpus_merged.jsonl'
TRAIN_OUTPUT=ROOT/'persian_corpus_merged_train.jsonl'
VALIDATION_OUTPUT=ROOT/'persian_corpus_merged_validation.jsonl'
OUTPUT_MANIFEST=ROOT/'persian_corpus_merged_manifest.json'
SEED=42
VALIDATION_FRACTION=0.01
MAX_ROWS_PER_SOURCE=None  # Optional positive integer for a controlled pilot.
if not SOURCES: raise ValueError('Configure at least one source.')
if MAX_ROWS_PER_SOURCE is not None and (not isinstance(MAX_ROWS_PER_SOURCE,int) or MAX_ROWS_PER_SOURCE<=0):
    raise ValueError('MAX_ROWS_PER_SOURCE must be None or a positive integer.')

## 3. Verify source provenance, licenses, and byte integrity

In [ ]:
verified=[]
for source in SOURCES:
    data_path=Path(source['data']); manifest_path=Path(source['manifest'])
    if not data_path.is_file() or not manifest_path.is_file():
        raise FileNotFoundError(f"Missing source artifact or manifest for {source['key']}")
    manifest=load_dataset_manifest(manifest_path)
    validate_dataset_manifest(manifest,intended_purpose='pretraining')
    verify_dataset_artifact(data_path,manifest,hash_field='combined_jsonl_sha256')
    verified.append({'key':source['key'],'data':data_path,'manifest_path':manifest_path,
                     'manifest':manifest,'manifest_sha256':sha256_file(manifest_path),
                     'data_sha256':sha256_file(data_path)})
print([{'key':x['key'],'dataset_id':x['manifest']['dataset_id'],
        'revision':x['manifest']['dataset_revision']} for x in verified])

## 4. Stream, normalize, deduplicate, and assign deterministic splits

In [ ]:
def stable_split(digest):
    bucket=int(hashlib.sha256(f'{SEED}:{digest}'.encode('ascii')).hexdigest()[:16],16)
    return 'validation' if bucket<int(VALIDATION_FRACTION*(1<<64)) else 'train'

SEEN_DB=ROOT/'merge_seen.sqlite'
if SEEN_DB.exists(): SEEN_DB.unlink()
seen_db=sqlite3.connect(SEEN_DB)
seen_db.execute('PRAGMA journal_mode=WAL')
seen_db.execute('CREATE TABLE seen (digest TEXT PRIMARY KEY)')
counts={x['key']:{'read':0,'kept':0,'duplicate':0,'rejected':0} for x in verified}
train_count=validation_count=0
try:
  with OUTPUT.open('w',encoding='utf-8') as combined, TRAIN_OUTPUT.open('w',encoding='utf-8') as train, VALIDATION_OUTPUT.open('w',encoding='utf-8') as validation:
    for source in verified:
        source_key=source['key']; limit=MAX_ROWS_PER_SOURCE
        for row_index,record in enumerate(iter_jsonl(source['data'])):
            if limit is not None and row_index>=limit: break
            counts[source_key]['read']+=1
            text_value=record.get('text')
            if not isinstance(text_value,str) or not text_value.strip():
                counts[source_key]['rejected']+=1; continue
            normalized=normalize_persian_text(text_value).strip()
            if len(normalized)<20:
                counts[source_key]['rejected']+=1; continue
            digest=content_sha256(normalized)
            try:
                seen_db.execute('INSERT INTO seen(digest) VALUES (?)',(digest,))
            except sqlite3.IntegrityError:
                counts[source_key]['duplicate']+=1; continue
            split=stable_split(digest)
            output_record={'source_key':source_key,'source_id':source['manifest']['dataset_id'],
                           'source_revision':source['manifest']['dataset_revision'],
                           'source_row':record.get('source_row',row_index),
                           'text':normalized,'sha256':digest,'split':split}
            line=json.dumps(output_record,ensure_ascii=False)+'\n'
            combined.write(line)
            if split=='train': train.write(line); train_count+=1
            else: validation.write(line); validation_count+=1
            counts[source_key]['kept']+=1
  seen_db.commit()
finally:
  seen_db.close()
if train_count==0 or validation_count==0:
    raise ValueError('Merged corpus has an empty train or validation split; add more source data or adjust validation fraction.')
NEAR_DUP_SAMPLE_ROWS=10000
near_sample=list(islice(iter_jsonl(OUTPUT),NEAR_DUP_SAMPLE_ROWS))
near_audit=find_near_duplicate_pairs(near_sample,id_field='source_row')
NEAR_DUP_REPORT=ROOT/'near_duplicate_candidates.json'
NEAR_DUP_REPORT.write_text(json.dumps(near_audit,ensure_ascii=False,indent=2),encoding='utf-8')
print({'train':train_count,'validation':validation_count,'sources':counts,
       'near_duplicate_sample_size':len(near_sample),'near_duplicate_candidates':len(near_audit['pairs'])})

## 5. Export a source-linked manifest

In [ ]:
source_manifest_hashes=sorted(x['manifest_sha256'] for x in verified)
revision='sha256:'+hashlib.sha256(''.join(source_manifest_hashes).encode('ascii')).hexdigest()
manifest={
    'created_at_utc':datetime.now(timezone.utc).isoformat(),
    'dataset_id':'local/davinci-merged-pretraining-corpus',
    'dataset_revision':revision,
    'purpose':'pretraining',
    'license_reviewed_by_user':all(x['manifest']['license_reviewed_by_user'] is True for x in verified),
    'license_approved_for_use':all(x['manifest']['license_approved_for_use'] is True for x in verified),
    'license_review_notes':'Each source manifest was individually reviewed and approved; see source_manifests for immutable provenance.',
    'source_manifests':[{'key':x['key'],'dataset_id':x['manifest']['dataset_id'],
        'dataset_revision':x['manifest']['dataset_revision'],'manifest_sha256':x['manifest_sha256'],
        'data_sha256':x['data_sha256'],'dataset_card_url':x['manifest'].get('dataset_card_url'),
        'license_declared_by_hub':x['manifest'].get('license_declared_by_hub'),'license_review_notes':x['manifest']['license_review_notes']} for x in verified],
    'seed':SEED,'validation_fraction':VALIDATION_FRACTION,
    'split_method':'SHA-256(seed:normalized_text_sha256) threshold; deterministic document-level split',
    'counts':counts,'train_rows':train_count,'validation_rows':validation_count,
    'combined_jsonl_sha256':sha256_file(OUTPUT),
    'train_jsonl_sha256':sha256_file(TRAIN_OUTPUT),
    'validation_jsonl_sha256':sha256_file(VALIDATION_OUTPUT),
    'normalization':'NFC and canonical Persian normalization via src.davinci.data_audit.normalize_persian_text',
    'deduplication':'Exact normalized-text SHA-256 across all input sources',
    'near_duplicate_audit':{'sample_size':len(near_sample),'candidate_pairs':len(near_audit['pairs']),
        'report_path':str(NEAR_DUP_REPORT),'method':near_audit['method'],'warning':near_audit['warning']},
    'known_limits':['No near-duplicate detection','No language identification','No quality scoring beyond non-empty/minimum length','Human review remains required'],
}
if manifest['license_reviewed_by_user'] is not True or manifest['license_approved_for_use'] is not True:
    raise PermissionError('At least one source manifest is not explicitly reviewed and approved.')
OUTPUT_MANIFEST.write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
validate_dataset_manifest(manifest,intended_purpose='pretraining')
verify_dataset_artifact(OUTPUT,manifest,hash_field='combined_jsonl_sha256')
print('Merged outputs:',OUTPUT,TRAIN_OUTPUT,VALIDATION_OUTPUT,OUTPUT_MANIFEST,sep='\n')

## 6. Review before training

Inspect samples from every source, compare language balance, inspect rejected/duplicate counts, and run near-duplicate, language-identification, quality, contamination, and privacy audits before full pretraining. The merge notebook performs exact deduplication only. It does not establish that the source data is lawful, representative, high quality, or free of sensitive content.